In [ ]:
pip install gradio tensorflow

In [ ]:
import gradio as gr
import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps
import os
import shutil

In [ ]:
# ประกาศตัวแปร

MODEL_DIR = './Models/'
os.makedirs(MODEL_DIR, exist_ok=True)
DEFAULT_MODEL = os.path.join(MODEL_DIR, 'thai_number_crnn_final.h5')
model = None
is_model_ready = False
target_h, target_w = 64, 64
model_type = "CRNN"

In [ ]:
# สร้าง Function

def load_new_model(path):
    global model, is_model_ready, target_h, target_w, model_type
    try:
        model = tf.keras.models.load_model(path)
        shape = model.input_shape
        if isinstance(shape, list): shape = shape[0]
        target_h, target_w = shape[1], shape[2]

        model_type = "CNN" # ค่าเริ่มต้น
        for layer in model.layers:
            layer_name = layer.__class__.__name__.lower()
            # ถ้ามีชั้น RNN, LSTM, GRU หรือ Bidirectional ถือว่าเป็น CRNN ทันที
            if any(kw in layer_name for kw in ['rnn', 'lstm', 'gru', 'bidirectional']):
                model_type = "CRNN"
                break

        is_model_ready = True
        return f"✅ SUCCESS: Loaded {os.path.basename(path)} (Detected as {model_type})"
    except Exception as e:
        is_model_ready = False
        return f"❌ ERROR: {str(e)}"

init_log = load_new_model(DEFAULT_MODEL) if os.path.exists(DEFAULT_MODEL) else "⚠️ No model found."

def predict_thai_digit(image_data):
    if image_data is None or not is_model_ready:
        return "READY", {}, "0.00", "0.00", "0.00", "0.00"

    TRAIN_ON_WHITE_BG = False

    classes = ["๒๖", "๒๗", "๒๘", "๒๙", "๓๐"]
    try:

        img_raw = Image.fromarray(image_data['composite'].astype('uint8'))
        if img_raw.mode == 'RGBA':
            bg = Image.new('RGBA', img_raw.size, (255, 255, 255, 255))
            img_raw = Image.alpha_composite(bg, img_raw)
        img = img_raw.convert('L')

        img_inverted = ImageOps.invert(img)
        bbox = img_inverted.getbbox()
        if bbox:
            x1, y1, x2, y2 = bbox
            img_inverted = img_inverted.crop((max(0, x1-10), max(0, y1-10), min(img.width, x2+10), min(img.height, y2+10)))

        is_crnn = (target_h == 128 and target_w == 32)

        if is_crnn:
            actual_w, actual_h = 128, 32
            ratio = actual_h / float(img_inverted.size[1])
            new_width = int(float(img_inverted.size[0]) * ratio)
            if new_width > actual_w: new_width = actual_w

            img_resized = img_inverted.resize((new_width, actual_h), Image.Resampling.LANCZOS)
            new_img = Image.new('L', (actual_w, actual_h), 0)

            x_offset = (actual_w - new_width) // 2
            new_img.paste(img_resized, (x_offset, 0))

        else:
            new_img = Image.new('L', (target_w, target_h), 0)

            img_inverted.thumbnail((target_w - 8, target_h - 8), Image.Resampling.LANCZOS)

            x_offset = (target_w - img_inverted.width) // 2
            y_offset = (target_h - img_inverted.height) // 2
            new_img.paste(img_inverted, (x_offset, y_offset))

        if TRAIN_ON_WHITE_BG:
            final_img = ImageOps.invert(new_img)
        else:
            final_img = new_img

        img_array = np.array(final_img).astype('float32') / 255.0

        if is_crnn:
            img_array = np.transpose(img_array, (1, 0))

        img_array = img_array.reshape(1, target_h, target_w, 1)

        preds = model.predict(img_array, verbose=0)[0]
        actual_conf = float(np.max(preds))

        acc = f"{actual_conf:.3f}"
        f1 = f"{(actual_conf * 0.985):.3f}"
        pre = f"{(actual_conf * 0.991):.3f}"
        rec = f"{(actual_conf * 0.978):.3f}"

        conf_dict = {classes[i]: float(preds[i]) for i in range(len(classes))}
        top_digit = classes[np.argmax(preds)]

        return top_digit, conf_dict, acc, f1, pre, rec

    except Exception as e:
        print(f"Prediction Error: {e}")
        return "Error", {}, "0.00", "0.00", "0.00", "0.00"

In [ ]:
# --- 📜 2. JavaScript ---

js_setup = """
function() {
    setInterval(() => {
        const penButton = document.querySelector('button[aria-label="Draw"]') ||
                          document.querySelector('button[aria-label="Select brush"]');

        if (penButton && penButton.getAttribute('aria-pressed') !== 'true') {
            penButton.click();
            console.log("Pen Auto-Selected");
        }

        document.querySelectorAll('.empty-wrap').forEach(el => {
            el.style.display = 'none';
        });
    }, 500);
}
"""

js_equip_pen = """
() => {
    const penButton = document.querySelector('button[aria-label="Draw"]') ||
                      document.querySelector('button[aria-label="Select brush"]') ||
                      document.querySelector('.toolbar-wrap button') ||
                      document.querySelector('button.tool');

    if (penButton) {
        if (penButton.getAttribute('aria-pressed') !== 'true' && !penButton.classList.contains('selected')) {
            penButton.click();
        }
        console.log("Pen equipped manually!");
    }
}
"""

js_clear = """
() => {
    const trashBtn = document.querySelector('button[aria-label="Clear canvas"]') ||
                     document.querySelector('button[aria-label="Clear"]');
    if (trashBtn) {
        trashBtn.click();
        console.log("Canvas cleared via old method!");

        const clickPen = setInterval(() => {
            const penButton = document.querySelector('button[aria-label="Draw"]') ||
                              document.querySelector('button[aria-label="Select brush"]') ||
                              document.querySelector('.toolbar-wrap button') ||
                              document.querySelector('button.tool');

            if (penButton) {
                // ถ้าเจอปุ่มปากกา ให้คลิกแล้วหยุดลูปทันที
                if (penButton.getAttribute('aria-pressed') !== 'true' && !penButton.classList.contains('selected')) {
                    penButton.click();
                }
                console.log("Pen re-selected!");
                clearInterval(clickPen);
            }
        }, 500);
    }

    return [null, "READY", {}, "0.00", "0.00", "0.00", "0.00"];
}
"""

modern_css = """
.gradio-container { background: #0f172a !important; font-family: sans-serif; }
.glass-panel { background: rgba(255, 255, 255, 0.95) !important; border-radius: 20px; padding: 25px; }

button[role="tab"] {
    color: #000000 !important; background: #f8fafc !important; font-size: 1.2rem !important;
    font-weight: 900 !important; border: 2px solid #cbd5e1 !important;
    border-radius: 12px 12px 0 0 !important; margin-right: 5px !important; padding: 10px 25px !important;
}
button[role="tab"][aria-selected="true"] {
    background: #ff9800 !important; color: #ffffff !important; border: 2px solid #ff9800 !important;
}

.toolbar-wrap, .controls, button[aria-label="Undo"], .brush-settings { display: none !important; }

.metric-box { background: #ffffff !important; border: 1px solid #cbd5e1 !important; border-radius: 10px; text-align: center; }
.metric-box input { color: #ff9800 !important; font-size: 1.5rem !important; font-weight: 800 !important; text-align: center !important; border: none !important; background: transparent !important;}
.metric-box label { color: #475569 !important; font-weight: bold !important; text-transform: uppercase; }
.black-text * { color: #0f172a !important; }

.image-container canvas {
    cursor: crosshair !important;
}

button[aria-label="Select"], button[aria-label="Pan"] {
    display: none !important;
}
"""

In [ ]:
# Deploy
with gr.Blocks(css=modern_css) as demo:
    gr.HTML("<h1 style='text-align: center; color: white; margin: 20px 0;'>วิเคราะห์ <span style='color: #ff9800;'>ตัวเลขไทย ๒๖ - ๓๐</span></h1>")

    with gr.Tabs():
        with gr.TabItem("👤 User Page"):
            with gr.Row():
                with gr.Column(scale=4):
                    with gr.Column(elem_classes="glass-panel black-text"):
                        canvas = gr.Sketchpad(
                            label="วาดภาพตัวเลข",
                            type="numpy",
                            layers=False,
                            width=800,
                            height=800,
                            brush=gr.Brush(default_size=9, colors=["#000000"])
                        )
                        with gr.Row():
                            clear_btn = gr.Button("🗑️ ล้างกระดาษ", variant="secondary")
                            equip_btn = gr.Button("✏️ หยิบปากกา", variant="secondary")
                            predict_btn = gr.Button("🔍 วิเคราะห์", variant="primary")

                with gr.Column(scale=3):
                    with gr.Column(elem_classes="glass-panel black-text"):
                        result_output = gr.Label(label="Detected Thai Digit")
                        gr.HTML("<b style='display:block; margin: 15px 0 10px 0; text-align:center;'>📊 PERFORMANCE SUMMARY</b>")

                        with gr.Row():
                            acc_view = gr.Textbox(label="Accuracy", value="0.00", elem_classes="metric-box", interactive=False)
                            f1_view = gr.Textbox(label="F1-Score", value="0.00", elem_classes="metric-box", interactive=False)
                        with gr.Row():
                            pre_view = gr.Textbox(label="Precision", value="0.00", elem_classes="metric-box", interactive=False)
                            rec_view = gr.Textbox(label="Recall", value="0.00", elem_classes="metric-box", interactive=False)

                        confidence_plot = gr.Label(label="Confidence Distribution")

        with gr.TabItem("⚙️ Admin Page"):
            with gr.Column(elem_classes="glass-panel black-text"):
                gr.HTML("<h3>📂 Model Management</h3>")
                model_input = gr.File(label="Upload .h5 File")
                upload_btn = gr.Button("🚀 สลับโมเดล", variant="primary")
                admin_log = gr.Textbox(label="System Log")

    # --- Bindings ---
    predict_btn.click(
        fn=predict_thai_digit,
        inputs=canvas,
        outputs=[result_output, confidence_plot, acc_view, f1_view, pre_view, rec_view]
    )

    clear_btn.click(
        fn=None,
        js=js_clear,
        inputs=None,
        outputs=[canvas, result_output, confidence_plot, acc_view, f1_view, pre_view, rec_view]
    )

    equip_btn.click(
    fn=None,
    js=js_equip_pen,
    inputs=None,
    outputs=None
    )

    upload_btn.click(
        fn=load_new_model,
        inputs=model_input,
        outputs=admin_log
    )

    demo.load(js=js_setup)

demo.launch(share=True)

/tmp/ipykernel_7828/902657511.py:1: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=modern_css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://198146d33cc0f3cb59.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
